### 任务 1.1：基础设置和基础智能体创建

使用 Strands 智能体框架设置客户支持智能体原型。该原型是您探索从智能体原型到生产就绪解决方案的完整旅程的起点。

完成此任务后，智能体将具有以下基础架构：

<div style="text-align:left">
    <img src="images/architecture_lab1_strands_zh_cn.png" width="75%"/>
</div>

*图片描述：使用本地工具在本地运行的简单智能体原型*

安装依赖项并导入所有必要的库（包括 AWS SDK、AgentCore 组件和 Strands 框架），以准备开发环境。

In [ ]:
import boto3
import json
import uuid
import time
import requests
from boto3.session import Session

# AgentCore imports
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

# Strands imports
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

# Local tools
from lab_helpers.lab1_strands_agent import (
    get_product_info, get_return_policy, get_technical_support, web_search,
    SYSTEM_PROMPT, MODEL_ID
)
from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter
from scripts.utils import get_cognito_client_secret

# Setup
boto_session = Session()
REGION = "us-east-1"
CUSTOMER_ID = "customer_001"
SESSION_ID = str(uuid.uuid4())

print("✅ Libraries imported successfully!")

在创建智能体之前，请检查将为我们的客户支持功能提供助力的本地工具。打开并查看 `lab_helpers/lab1_strands_agent.py` 以了解：

- 工具在此文件中使用 `@tool` 装饰器在本地定义
- 四个工具函数及其用途：
  - get_product_info()：获取产品信息
  - get_return_policy()：获取特定产品的退货政策
  - get_technical_support()：提供技术支持指导
  - web_search()：在 Web 上搜索更新的信息
- 它们如何使用模拟数据（模拟真实的数据库/API）
- 定义智能体行为的系统提示词

创建基础客户支持智能体，以演示核心AI功能，从理解查询到执行操作。该智能体结合了：

- **基础模型**：推动推理和决策的“大脑”
- **系统提示词**：定义智能体个性和服务标准的行为指令
- **专业工具**：四种本地工具（产品信息、退货政策、技术支持、Web 搜索）

当您调用智能体时，智能体将遵循以下流程：
1. **查询分析**：智能体分析客户的问题
2. **工具选择**：智能体决定使用哪些工具（如果有）
3. **工具执行**：智能体使用正确的参数调用相应的工具
4. **响应合成**：智能体将工具结果与自身知识相结合，以创建有用的响应
5. **质量检查**：智能体确保响应符合系统提示词中的标准

In [ ]:
# Create a basic agent with local tools
model = BedrockModel(model_id=MODEL_ID, temperature=0.3, region_name=REGION)
basic_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Basic customer support agent ready!")
print("📋 Available tools: Product Info, Return Policy, Technical Support, Web Search")

测试基础智能体，看看它如何处理客户查询和使用其工具：

In [ ]:
# Test basic agent functionality
print("💬 Testing basic agent...\n")
response = basic_agent("What's the return policy for laptops?")
print("\n" + "="*50 + "\n")

这个初始原型有一些限制，您将在后续任务中解决这些限制：

- **没有持久记忆** – 智能体会忘记之前会话中的客户历史记录和偏好
- **仅限本地工具** – 未集成共享或企业级工具  
- **没有身份管理** – 无法代表特定用户行事

### 任务 1.2：通过记忆功能增强智能体

一位尊贵的客户就最近的订单问题联系了您的支持团队。他们说明了自己的偏好，分享了他们的沮丧，并与您的智能体合作解决问题。三周后，他们再次就相关问题联系支持团队。但是，这时他们必须重复所有内容（他们的偏好、历史记录和上下文），因为您的智能体只记得当前的对话会话，而不记得之前的会话。这会导致：
- **客户感到沮丧**，他们必须反复重复他们的信息
- **支持效率低下**，无法基于之前的交互提供支持
- **客户满意度不佳**，获得非个人化、笼统的响应

Amazon Bedrock AgentCore Memory 通过提供一项托管服务来解决这一限制，该托管服务使AI 智能体能够长久保持上下文、记住重要事实并提供一致的个性化体验。AgentCore Memory 在两个级别上运行：
- **短期记忆**：即时对话上下文和基于会话的信息（由 Strands 智能体框架自动处理）
- **长期记忆**：在多个对话中提取的持久信息，包括事实、偏好和摘要（通过采用 USER_PREFERENCE 和 SEMANTIC 策略的 AgentCore Memory 服务实现）

将原型转换为具有客户意识的助手，该助手能够执行以下示例操作：
- **“欢迎回来，Sarah！”**– 即时识别回头客
- **“跟进上个月的笔记本电脑问题”**– 无缝关联相关对话
- **“根据您的购买历史记录，为您推荐以下内容”**– 提供个性化推荐

完成此任务后，智能体将具有以下集成记忆功能的架构：

<div style="text-align:left">
    <img src="images/architecture_lab2_memory_zh_cn.png" width="75%"/>
</div>

*图片描述：使用 AgentCore Memory 增强智能体，AgentCore Memory 可提供持久的客户上下文和个性化体验*

**记忆策略配置**：创建结合两种智能策略的记忆资源：

| 策略类型 | 用途 | 客户利益 |
|---------------|---------|------------------|
| USER_PREFERENCE | 客户的偏好和行为 |“我记得您更喜欢...”|
| SEMANTIC | 事实信息和上下文 |“关于您之前的问题...”|

AgentCore Memory 通过命名空间使用 actorId 对长期记忆消息进行逻辑分组：
- `support/customer/{actorId}/preferences`：用于用户偏好记忆策略
- `support/customer/{actorId}/semantic`：用于语义记忆策略

In [ ]:
# Initialize memory client for AgentCore Memory service
memory_client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"

def create_or_get_memory_resource():
    try:
        # Try to get existing memory resource from SSM parameter
        memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        return memory_id
    except:
        # Create new memory resource with two strategies
        strategies = [
            {
                # USER_PREFERENCE strategy captures customer preferences and behaviors
                StrategyType.USER_PREFERENCE.value: {
                    "name": "CustomerPreferences",
                    "description": "Captures customer preferences and behavior",
                    "namespaces": ["support/customer/{actorId}/preferences"],
                }
            },
            {
                # SEMANTIC strategy stores factual information from conversations
                StrategyType.SEMANTIC.value: {
                    "name": "CustomerSupportSemantic",
                    "description": "Stores facts from conversations",
                    "namespaces": ["support/customer/{actorId}/semantic"],
                }
            },
        ]
        print("Creating AgentCore Memory resources (2-3 minutes)...")
        # Create memory resource and wait for completion
        response = memory_client.create_memory_and_wait(
            name=memory_name,
            description="Customer support agent memory",
            strategies=strategies,
            event_expiry_days=90,  # Memory events expire after 90 days
        )
        memory_id = response["id"]
        # Store memory ID in SSM for future use
        put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)
        return memory_id

memory_id = create_or_get_memory_resource()
print(f"✅ Memory resource ready: {memory_id}")

模拟一位名为“customer_001”的回头客，该客户之前曾与您的支持团队进行过互动。这将演示 AgentCore Memory 如何自动将单个对话转化为丰富、持久的客户见解。加载之前的客户交互，观看 AgentCore Memory 如何自动将其转化为长期的客户见解。

In [ ]:
# Seed previous customer interactions
previous_interactions = [
    ("I'm having issues with my MacBook Pro overheating during video editing.", "USER"),
    ("I can help with that thermal issue. Your MacBook Pro order #MB-78432 is still under warranty.", "ASSISTANT"),
    ("What's the return policy on gaming headphones? I need low latency for competitive FPS games", "USER"),
    ("For gaming headphones, you have 30 days to return. Since you're into competitive FPS, I'd recommend checking audio latency specs.", "ASSISTANT"),
    ("I need a laptop under $1200 for programming. Prefer 16GB RAM minimum and good Linux compatibility. I like ThinkPad models.", "USER"),
    ("Perfect! For development work, I'd suggest ThinkPad E series or Dell XPS models with excellent Linux support.", "ASSISTANT"),
]

if memory_id:
    memory_client.create_event(
        memory_id=memory_id,
        actor_id=CUSTOMER_ID,
        session_id="previous_session",
        messages=previous_interactions
    )
    print("✅ Customer history seeded successfully")
    print("⏳ Long-term memory processing will begin automatically...")

Strands 智能体提供强大的钩子系统，使组件能够通过强类型的事件回调来对智能体行为做出反应或修改智能体行为。这可确保记忆操作自动发生，无需手动干预。

每次客户与智能体进行交互时，智能体将自动：
- 根据客户之前的交互和偏好**提供个性化对话**
- **向记忆中添加新的交互**以持续改善未来的个性化体验

钩子集成的作用：
- **响应前**：自动检索相关的客户上下文和偏好
- **响应后**：自动将新的交互保存到 AgentCore Memory 中

使用记忆钩子启用自动客户上下文。

In [ ]:
class CustomerSupportMemoryHooks(HookProvider):
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        # Hook that runs before agent responds to retrieve customer context
        messages = event.agent.messages
        # Only process user messages (not tool results)
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                all_context = []
                # Retrieve memories from each strategy namespace, both USER_PREFERENCE and SEMANTIC
                for context_type, namespace in self.namespaces.items():
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,  # Get top 3 relevant memories
                    )
                    # Extract text content from memory objects
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(f"[{context_type.upper()}] {text}")
                
                # Prepend customer context to user message
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = f"Customer Context:\n{context_text}\n\n{original_text}"
            except Exception as e:
                print(f"Failed to retrieve customer context: {e}")

    def save_support_interaction(self, event: AfterInvocationEvent):
        # Hook that runs after agent responds to save interaction to memory
        try:
            messages = event.agent.messages
            # Only save if we have both user and assistant messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                customer_query = None
                agent_response = None
                
                # Find the most recent user query and assistant response
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                # Save the interaction to AgentCore Memory
                if customer_query and agent_response:
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[(customer_query, "USER"), (agent_response, "ASSISTANT")],
                    )
        except Exception as e:
            print(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        # Register both hooks with the agent's hook registry
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)

print("✅ Memory hooks defined - Automatic customer personalization enabled!")
print("🧠 Your agent will now remember customers and personalize every interaction")

创建并测试记忆增强的智能体，看看它如何检索客户上下文并提供个性化响应：

In [ ]:
# Create memory-enhanced agent with hooks
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

memory_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    hooks=[memory_hooks],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Memory-enhanced agent created!")
print("🧠 Agent will automatically retrieve customer context and save interactions")

In [ ]:
# Wait for memory processing to complete
print("⏳ Waiting 90 seconds for memory processing to complete...")
time.sleep(90)

# Test memory recall
print("🧠 Testing memory-enhanced agent...\n")
response = memory_agent("What are my laptop preferences?")
print("\n" + "="*50 + "\n")

### 任务 1.3：通过网关集成和 AgentCore Identity 进行扩展

实现记忆功能后，专注于强大的工具并扩大其影响。优秀的智能体需要充分利用专有及第三方 API 和数据的工具，这样这些智能体才能为内部和外部客户完成工作。但是，构建、保护和扩展智能体工具很困难，这成为客户在生产环境中从智能体原型向实际智能体商业价值转变过程中的重大障碍。AgentCore Gateway 充当连接层，使AI 智能体能够通过统一的**模型上下文协议 (MCP)** 终端节点对现实世界的工具进行发现、身份验证和调用。这对于管理数百个 API、资源和工具的企业至关重要。

主要优势：
- **完全托管式 MCP 服务器**解决方案，无需基础设施管理
- **集成现有 API** 和 Lambda 函数
- 集中管理不同工具的**统一界面**
- **安全的身份验证**和授权
- **语义工具的发现**和选择

##### 您要构建的内容：

**工具集中化和可重用性**：
- 将 Web 搜索从本地工具迁移到集中式 AgentCore Gateway
- 集成现有的企业 Lambda 函数（保修检查）
- 创建多个智能体类型可以访问的共享工具基础设施

**企业级安全**：
- 通过 Cognito 集成实施基于 JWT 的身份验证
- 为网关访问配置安全的入站授权
- 为工具使用建立基于身份的访问控制

这奠定了可扩展的基础，工具可以集中进行管理并在多种智能体类型中重复使用，从而消除代码重复并简化维护。

AgentCore Identity 也参与了这个过程。它使AI 智能体能够安全地访问 AWS 资源，并通过与 Amazon Cognito 协作来帮助处理入站调用者身份验证。它还使AI 智能体能够使用出站身份验证安全地访问第三方工具和服务，但本实验中未使用 AgentCore Identity 的这一功能。

<div style="text-align:left">
    <img src="images/architecture_lab3_identity_zh_cn.png" width="75%"/>
</div>

完成此任务后，智能体将具有以下集成网关功能的架构：

<div style="text-align:left">
    <img src="images/architecture_lab3_gateway_zh_cn.png" width="75%"/>
</div>

*图片描述：使用 AgentCore Gateway 增强智能体，AgentCore Gateway 可实现安全、集中化的工具管理和企业集成*

创建 AgentCore Gateway 以将 Lambda 函数作为与 MCP 兼容的终端节点公开。要验证有权调用工具的调用者，请使用 OAuth 授权（MCP 服务器的标准）配置**入站身份验证**。

### 了解 Gateway 身份验证

AgentCore Gateway 使用**带有 JWT 令牌的 OAuth 2.0** 来安全访问工具。这可以防止未经授权的应用程序调用 Lambda 函数。

**重要概念**：

1. **身份验证提供商**：Amazon Cognito 管理身份并颁发令牌
2. **客户端证书**：智能体使用 client_id 和 client_secret（例如应用程序的用户名/密码）
3. **JWT 令牌**：证明智能体已获得授权的短期令牌
4. **允许的客户端**：可以访问 Gateway 的客户端 ID 的允许列表

**工作原理**：
```
智能体 → Cognito：“这是我的 client_id 和 client_secret”
Cognito → 智能体：“这是您的 JWT 访问令牌”
智能体 → Gateway：“这是我的令牌”
Gateway → Cognito：“这个令牌是否有效且来自允许的客户端？”
Gateway → 智能体：“已授予访问权限”
```

**安全注意事项**：这些证书已为您预先创建并安全地保存在 SSM Parameter Store 中。请勿在代码中对凭证进行硬编码

In [ ]:
# Retrieve authentication configuration from SSM Parameter Store
# These values were created by the CloudFormation template

# Client ID: Identifies which application is making the request
machine_client_id = get_ssm_parameter("/app/customersupport/agentcore/machine_client_id")
print(f"Machine Client ID: {machine_client_id}")

# Discovery URL: Tells the Gateway where to find Cognito's OAuth configuration
# This URL provides metadata about token endpoints, supported scopes, etc.
cognito_discovery_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
print(f"Discovery URL: {cognito_discovery_url}")

# Configure JWT-based authentication for the Gateway
auth_config = {
    "customJWTAuthorizer": {
        # Only tokens from this client ID will be accepted
        "allowedClients": [machine_client_id],
        # Gateway will fetch OAuth metadata from this URL
        "discoveryUrl": cognito_discovery_url
    }
}

print("✅ Authentication configuration ready")

###创建 AgentCore Gateway

Gateway 充当智能体和后端 Lambda 函数之间的安全智能体。可以将其视为专为AI 智能体设计的 API 网关。

**我们要构建的内容**：
- **Gateway 基础设施**：核心 Gateway 资源
- **MCP 协议**：工具通信的标准协议
- **JWT 授权**：使用我们刚刚创建的身份验证配置
- **IAM 角色**：调用 Lambda 函数的权限

**后续操作**：
1. 创建 Gateway（此单元格）
2. 使用工具定义添加 Lambda 目标（下一个单元格）
   - 一个 Lambda 函数可处理多个工具：`check_warranty_status` 和 `web_search`
   - Gateway 将工具名称传递给 Lambda，Lambda 会路由到相应的处理程序
3. 将智能体连接到 Gateway

**注意**：CloudFormation 模板部署了一个可以处理多项工具操作的单个 Lambda 函数 (`CustomerSupportLambda`)。这比为每个工具部署单独的 Lambda 函数更有效。

In [ ]:
class CreationFailedError(Exception):
    def __init__(self, message):
        self.message = message
        super().__init__(self.message)

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
gateway_name = "customersupport-gw"

try:
    print(f"Creating gateway: {gateway_name}")
    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
        protocolType="MCP",  # Model Context Protocol
        authorizerType="CUSTOM_JWT",  # Use JWT tokens for auth
        authorizerConfiguration=auth_config,  # Our auth config from above
        description="Customer Support AgentCore Gateway",
    )
    gateway_id = create_response["gatewayId"]
    gateway_url = create_response["gatewayUrl"]
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)

    # Wait for Gateway to be ready
    print("Waiting for Gateway to be ready...")
    while True:
        status = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['status']
        if status == 'READY':
            break
        elif status == 'FAILED':
            raise CreationFailedError("Gateway creation failed")
        else:
            print(f"  Status: {status}")
            time.sleep(5)

    print(f"✅ Gateway created successfully!")
    print(f"   Gateway ID: {gateway_id}")
    print(f"   Gateway URL: {gateway_url}")
    
except gateway_client.exceptions.ConflictException:
    # Gateway already exists, retrieve it
    gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = gateway_response["gatewayUrl"]
    print(f"✅ Using existing gateway: {gateway_id}")
    
except CreationFailedError:
    print("\033[31m❌ Gateway creation failed. Check CloudWatch logs for details.\033[0m")

AgentCore Gateway 使用要调用的工具的名称填充 Lambda 上下文，而传递给该工具的参数则在 Lambda 事件中提供。这使您可以集成现有的企业 Lambda 函数（在本例中为 `AgentCoreLab-CustomerSupportLambda`），这些函数可在多个智能体中重复使用。

使用 API 规范将 Lambda 函数添加为网关目标：

In [ ]:
# Load API specification for Lambda tools
api_spec = [
    {
        "name": "check_warranty_status",
        "description": "Check warranty status using serial number and email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "serial_number": {"type": "string"},
                "customer_email": {"type": "string"}
            },
            "required": ["serial_number"]
        }
    },
    {
        "name": "web_search",
        "description": "Search the web for updated information",
        "inputSchema": {
            "type": "object",
            "properties": {
                "keywords": {"type": "string", "description": "Search query keywords"},
                "region": {"type": "string", "description": "Search region (e.g., us-en)"},
                "max_results": {"type": "integer", "description": "Maximum results"}
            },
            "required": ["keywords"]
        }
    }
]

# Create gateway target
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": get_ssm_parameter("/app/customersupport/agentcore/lambda_arn"),
            "toolSchema": {"inlinePayload": api_spec},
        }
    }
}

try:
    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaTarget",
        description="Lambda tools for customer support",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    )
    print(f"✅ Gateway target created: {create_target_response['targetId']}")
except Exception as e:
    print(f"Gateway target may already exist: {str(e)}")

将 Cognito 的身份验证令牌集成到 Strands SDK 中的 MCPClient，以创建安全的 MCP 连接。

创建经过身份验证的 MCP 客户端来访问网关工具：

In [ ]:
def get_cognito_client_secret():
    # Get Cognito client secret using Cognito API
    client = boto3.client("cognito-idp")
    response = client.describe_user_pool_client(
        UserPoolId=get_ssm_parameter("/app/customersupport/agentcore/userpool_id"),
        ClientId=get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
    )
    return response["UserPoolClient"]["ClientSecret"]

def get_oauth_token():
    # Get OAuth token for gateway authentication using client credentials flow
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    data = {
        "grant_type": "client_credentials",  # OAuth 2.0 client credentials flow
        "client_id": get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
        "client_secret": get_cognito_client_secret(),
        "scope": get_ssm_parameter("/app/customersupport/agentcore/cognito_auth_scope"),
    }
    # Request access token from Cognito
    response = requests.post(
        get_ssm_parameter("/app/customersupport/agentcore/cognito_token_url"),
        headers=headers, data=data
    )
    return response.json()

# Get OAuth access token (JWT format)
token_response = get_oauth_token()
access_token = token_response['access_token']

# Create MCP client with Bearer token authentication
mcp_client = MCPClient(
    url=gateway_url,
    headers={"Authorization": f"Bearer {access_token}"},  # JWT token in Authorization header
)

print(f"✅ MCP client configured for gateway: {gateway_url}")

将所有东西组合在一起 – 记忆钩子 + 本地工具 + 网关工具。这将创建一个混合架构，其中一些工具仍然位于本地（用于实现快速和简化），而一些工具则通过网关进行集中管理（用于实现可重用性和企业集成）。

这种方法可消除不同智能体之间的代码重复，并为工具更新创建集中管理：

In [ ]:
# Initialize memory hooks for customer context
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

# Start MCP client connection to gateway
mcp_client.start()
# Retrieve available tools from the gateway
gateway_tools = mcp_client.list_tools_sync()

# Combine local tools with centralized gateway tools
all_tools = [
    get_product_info,      # Local tool
    get_return_policy,     # Local tool
    get_technical_support, # Local tool
] + gateway_tools          # Gateway tools - web_search, check_warranty_status

# Create enhanced agent with memory and gateway integration
enhanced_agent = Agent(
    model=model,
    tools=all_tools,           # Local + gateway tools
    hooks=[memory_hooks],      # Automatic memory operations
    system_prompt=SYSTEM_PROMPT
)

print("✅ Enhanced Customer Support Agent created!")
print(f"📊 Total tools available: {len(all_tools)}")
print(f"🧠 Memory enabled with ID: {memory_id}")
print(f"🔒 Secure gateway integration: {gateway_url}")

测试具有记忆和网关功能的智能体。验证智能体可以无缝地使用本地工具和集中式网关工具，同时通过记忆维护客户上下文。

测试场景包括保修检查、Web 搜索以及组合的记忆 + 网关功能：

In [ ]:
# Test gateway tools
print("🔍 Testing gateway web search...\n")
response2 = enhanced_agent("Search for latest iPhone 15 troubleshooting tips")
print("\n" + "="*50 + "\n")

In [ ]:
# Test warranty check
print("🛡️ Testing warranty check...\n")
response3 = enhanced_agent("Check warranty status for serial number ABC12345678")
print("\n" + "="*50 + "\n")

In [ ]:
# Test combined capabilities
print("🎯 Testing combined memory + gateway capabilities...\n")
response4 = enhanced_agent("I need gaming headphones again, and also search for the latest reviews")
print("\n" + "="*50 + "\n")

## 后续步骤

🎉 **恭喜**！ 您已完成 Notebook 练习。

您已成功完成以下任务：
- 使用 Strands 创建基础AI 智能体原型
- 使用 AgentCore Memory 对其进行增强，以提供持久的客户上下文
- 集成 AgentCore Gateway，实现安全、集中化的工具共享
- 测试完整的生产就绪客户支持系统

### 后续操作

1. **关闭此 Notebook 文件**
2. **返回至实验说明**
3. **继续执行任务 2**，在实际场景中浏览 AgentCore 控制平面并查看资源
